# 🚀 Biến Google Colab Thành Server Học Tập CI/CD (1-Click Run)
Notebook này tự động thiết lập toàn bộ môi trường Server trên Google Colab:
- **OpenSSH Server** lắng nghe tại  với đầy đủ driver GPU Tesla T4.
- **Tailscale VPN** kết nối an toàn với máy tính của bạn.
- **Docker & Docker Compose** sẵn sàng deploy container.
- **GitHub Actions Self-Hosted Runner** tự động chạy CI/CD.

### Bước 1: Giữ phiên làm việc (Keep-Alive JavaScript)
Bật script JavaScript giữ kết nối trình duyệt tránh bị Colab ngắt khi chuyển tab.

In [ ]:
from IPython.display import display, Javascript

def keep_alive_js():
    js_code = """
    function ClickConnect(){
        console.log("[Keep-Alive] Pinging Colab connection...");
        var btn = document.querySelector("colab-connect-button");
        if (btn) btn.click();
    }
    setInterval(ClickConnect, 60000);
    console.log("[Keep-Alive] Script đã kích hoạt (chu kỳ 60s)");
    """
    display(Javascript(js_code))

keep_alive_js()

### Bước 2: Tải và Cập nhật Mã Nguồn 

In [ ]:
import os
WORKSPACE_DIR = "/content/Colab2Server"
REPO_URL = "https://github.com/giangpvg/Colab2Server.git"

if not os.path.exists(WORKSPACE_DIR):
    print(">>> Đang tải mã nguồn từ GitHub...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print(">>> Đang cập nhật mã nguồn mới nhất...")
    !git -C {WORKSPACE_DIR} pull origin main 2>/dev/null || true

%cd {WORKSPACE_DIR}
!chmod +x scripts/*.sh scripts/*.py 2>/dev/null || true
print("
>>> Các script sẵn sàng:")
!ls -la scripts/

### Bước 3: Cấu Hình Thông Số (Điền Form Bên Phải ➡️)
> **Lưu ý quan trọng về Tailscale Auth Key**:
> Khi tạo key tại [login.tailscale.com/admin/settings/keys](https://login.tailscale.com/admin/settings/keys), **BẮT BUỘC TÍCH VÀO ** để key có thể dùng lại nhiều lần mỗi khi Colab khởi động lại!

In [ ]:
#@title ⚙️ Bảng Cấu Hình Dịch Vụ { form-width: "60%" }

# --- 1. Mạng & Tunnel (Tailscale Khuyên Dùng) ---
TUNNEL_MODE = "tailscale" #@param ["tailscale", "cloudflare-quick", "none"]
TAILSCALE_AUTH_KEY = "" #@param {type:"string"}

# --- 2. Gói CI: GitHub Actions Runner ---
ENABLE_RUNNER = True #@param {type:"boolean"}
GITHUB_REPO_URL = "https://github.com/giangpvg/Colab2Server" #@param {type:"string"}
RUNNER_TOKEN = "" #@param {type:"string"}
RUNNER_NAME = "colab-runner-01" #@param {type:"string"}
RUNNER_LABELS = "colab,gpu,cuda,staging" #@param {type:"string"}

# --- 3. Gói CD: Docker & OpenSSH Server ---
ENABLE_DOCKER = True #@param {type:"boolean"}
ENABLE_SSH = True #@param {type:"boolean"}
SSH_ROOT_PASSWORD = "colab123456" #@param {type:"string"}

print("Đã lưu thông số cấu hình!")

### Bước 4: Kích Hoạt Toàn Bộ Dịch Vụ (1-Click Run)
Nhấn nút Play để tự động cài đặt Docker, SSH, Tailscale và GitHub Runner!

In [ ]:
# 1. Khởi tạo hệ thống cơ bản
!bash scripts/setup_system.sh

# 2. Kích hoạt Docker Engine
if ENABLE_DOCKER:
    !bash scripts/setup_docker.sh

# 3. Kích hoạt SSH Server (Tự động mở 0.0.0.0:22 và gán GPU PATH)
if ENABLE_SSH:
    !bash scripts/setup_ssh.sh "{SSH_ROOT_PASSWORD}"

# 4. Kích hoạt Mạng Riêng Ảo Tailscale / Cloudflare Tunnel
if TUNNEL_MODE == "tailscale" and TAILSCALE_AUTH_KEY:
    !bash scripts/setup_tunnel.sh "tailscale" "{TAILSCALE_AUTH_KEY}"
elif TUNNEL_MODE == "cloudflare-quick":
    !bash scripts/setup_tunnel.sh "cloudflare-quick"
else:
    print("ℹ️ Chưa cấu hình key mạng (TAILSCALE_AUTH_KEY trống).")

# 5. Kích hoạt GitHub Actions Runner
if ENABLE_RUNNER and RUNNER_TOKEN:
    !bash scripts/setup_runner.sh "{GITHUB_REPO_URL}" "{RUNNER_TOKEN}" "{RUNNER_NAME}" "{RUNNER_LABELS}"
else:
    print("ℹ️ Bỏ qua kích hoạt Runner (RUNNER_TOKEN trống).")

### Bước 5: Giám Sát Tài Nguyên & Duy Trì Phiên (Dashboard)
Chạy cell này để duy trì kết nối và theo dõi nhiệt độ, VRAM GPU Tesla T4, RAM và CPU.

In [ ]:
!python3 scripts/keep_alive.py